# Time Series Autoencoder for Anomaly Detection

In this exercise we build an **LSTM Autoencoder** to detect anomalies in a multivariate time series. The key idea: train the autoencoder to reconstruct *normal* data, then flag windows with high reconstruction error as anomalies.

**Dataset:** `processminer-rare-event-mts.xlsx` -- 2-minute data from a pulp-and-paper machine, 1-29 May 1999 (18,398 rows). `y = 1` marks a sheet break (124 breaks, one row each); `x1`-`x61` are process variables. Downtime after most breaks has been removed, so the time stamps have gaps. Reading `.xlsx` files needs the `openpyxl` package.

**Learning objectives:**
1. Load and explore multivariate time series data from an Excel file.
2. Split in time: train on an early normal period, set the threshold on a later normal period, evaluate on the rest.
3. Build an LSTM autoencoder (encoder + decoder) and train it on normal training windows only.
4. Use reconstruction error to raise alerts.
5. Report events found, events missed and false alarms per week, not only F1.

**Evaluation design.** Training 1-14 May, validation 15-19 May (early stopping and the threshold), test 20-29 May, scored once. The scaler and the threshold are fitted on earlier periods only. Runtime: about 1.5 minutes on an Apple M3 Max laptop CPU, including about 10 s to read the Excel file; allow up to twice as long on older laptops.

---
## 1. Imports

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)  # initial weights and DataLoader shuffling

print(f"PyTorch version: {torch.__version__}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

---
## 2. Load and Explore the Data

In [ ]:
# YOUR CODE HERE
# Hint: Use pd.read_excel("processminer-rare-event-mts.xlsx")  (needs openpyxl)
# Print shape, columns, and head()

In [ ]:
# YOUR CODE HERE
# Hint: Use df.describe()

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. The label column is "y" (1 = sheet break) -- NOT the first column, which is the time stamp:
#    label_col = "y"; time_col = "time"
# 2. Print the value counts and the event rate
# 3. Check the sampling interval: step = df[time_col].diff(); print step.mode()[0]
#    and the number of longer gaps (step > step.mode()[0])

---
## 3. Visualize the Data

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. Sensor columns without the label, the time stamp, x28 (a grade code) and x61 (a flag):
#    feature_cols = [c for c in df.columns if c not in (label_col, time_col, "x28", "x61")]
# 2. anomaly_idx = df[df[label_col] == 1].index
# 3. Create subplots for the first 4 features
# 4. Plot each feature and scatter anomaly points in red

---
## 4. Split in Time and Standardize Features

Three consecutive periods: **training** (1-14 May) to fit the scaler and the autoencoder, **validation** (15-19 May) for early stopping and the threshold, and **test** (20-29 May), scored once at the end. The scaler is fitted on the normal rows of the training period only and then applied unchanged to every period.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. labels = df[label_col].to_numpy(); times = df[time_col]
#    features = df[feature_cols].to_numpy(dtype=np.float32)
# 2. Periods: TRAIN_END = pd.Timestamp("1999-05-15"); VAL_END = pd.Timestamp("1999-05-20")
#    period = np.select([(times < TRAIN_END).to_numpy(), (times < VAL_END).to_numpy()], [0, 1], default=2)
#    PERIODS = ["train", "validation", "test"]; print rows and breaks per period
# 3. Fit StandardScaler on NORMAL rows of the TRAINING period only:
#    scaler.fit(features[(period == 0) & (labels == 0)])
# 4. features_scaled = scaler.transform(features)   # every period, same fitted scaler

---
## 5. Create Sequences within Each Period

Windows are 30 rows long (60 minutes) and are labelled by their last row.

- **No window crosses a gap or a split.** A new stretch starts after every time gap (downtime) and at each period boundary, and windows are built only inside a stretch.
- **Normal windows** contain no break and do not end in the 30 minutes (15 rows) before a break or at it. The autoencoder trains on normal training windows only; normal validation windows are used for early stopping and for the threshold.
- **Per-window centring.** Several sensors drift between periods (the largest shifts are printed below, in training standard deviations). A level shift of many standard deviations would dominate the reconstruction error, and the detector would flag drift rather than unusual behaviour. We therefore subtract each window's own mean from every sensor: the autoencoder sees how the sensors move within the hour. This uses only the window itself, so it is available at every time step.

In [ ]:
SEQ_LENGTH = 30  # 30 rows = 60 minutes
LEAD_ROWS = 15   # an alert counts as a warning if it comes at most 30 minutes before a break

# Drift check on normal validation rows, in training standard deviations
val_shift = pd.Series(features_scaled[(period == 1) & (labels == 0)].mean(axis=0), index=feature_cols)
print("Largest mean shifts in the validation period (training SDs):")
print(val_shift.abs().sort_values(ascending=False).head(5).round(1).to_string())

# Contiguous stretches: a new stretch after every time gap and at every period boundary
new_stretch = (times.diff() > pd.Timedelta(minutes=2)).to_numpy() | (np.diff(period, prepend=period[0]) != 0)
stretch = np.cumsum(new_stretch)

# Window end rows: rows with at least SEQ_LENGTH - 1 earlier rows in the same stretch
ends = np.concatenate([np.flatnonzero(stretch == s)[SEQ_LENGTH - 1:] for s in np.unique(stretch)])


def create_sequences(data, ends, seq_length):
    """Windows of `seq_length` rows ending at each row in `ends`, centred per window and sensor."""
    # YOUR CODE HERE
    # Hint:
    # 1. windows = np.stack([data[e - seq_length + 1 : e + 1] for e in ends])
    # 2. Subtract each window's own mean per sensor:
    #    return windows - windows.mean(axis=1, keepdims=True)
    pass


sequences = create_sequences(features_scaled, ends, SEQ_LENGTH)

# Window categories (each window is labelled by its last row)
event_rows = np.flatnonzero(labels == 1)
contains_break = np.array([labels[e - SEQ_LENGTH + 1 : e + 1].any() for e in ends])
warning_period = np.zeros(len(ends), dtype=bool)
for e in event_rows:
    warning_period |= (ends >= e - LEAD_ROWS) & (ends <= e)
normal = ~contains_break & ~warning_period
window_period = period[ends]

print(f"\nSequences shape: {sequences.shape}")
for p, name in enumerate(PERIODS):
    in_p = window_period == p
    print(f"{name:>10}: {in_p.sum():5d} windows, {(in_p & normal).sum():5d} normal")

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. train_seq = sequences[(window_period == 0) & normal]; val_seq = sequences[(window_period == 1) & normal]
# 2. Convert to PyTorch tensors (float32): train_t, val_t
# 3. Create DataLoaders -- NOTE: for an autoencoder, input == target
#    TensorDataset(train_t, train_t)
# 4. BATCH_SIZE = 64 (shuffle=True for train, False for val)

---
## 6. Build LSTM Autoencoder

The autoencoder consists of:
- **Encoder**: LSTM that compresses the input sequence into a fixed-size hidden state.
- **Decoder**: LSTM that reconstructs the original sequence from the hidden state.

In [ ]:
class LSTMAutoencoder(nn.Module):
    """LSTM Autoencoder for time series anomaly detection."""

    def __init__(self, input_size, hidden_size=64, num_layers=1):
        super().__init__()
        # YOUR CODE HERE
        # Hint:
        # 1. self.encoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # 2. self.decoder = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        # 3. self.fc = nn.Linear(hidden_size, input_size)
        pass

    def forward(self, x):
        # YOUR CODE HERE
        # Hint:
        # 1. Encode: _, (hidden, cell) = self.encoder(x)
        # 2. Decode step-by-step:
        #    - Start with decoder_input = x[:, -1:, :]
        #    - Loop seq_len times:
        #      decoder_out, (hidden, cell) = self.decoder(decoder_input, (hidden, cell))
        #      out = self.fc(decoder_out)
        #      outputs.append(out)
        #      decoder_input = out
        # 3. Concatenate and flip: outputs = torch.flip(torch.cat(outputs, dim=1), dims=[1])
        pass


num_features = features_scaled.shape[1]
model = LSTMAutoencoder(input_size=num_features, hidden_size=64).to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 7. Train on Normal Data Only

The loss on normal validation windows decides when to stop. The best weights are kept as a deep copy and restored before any error is computed. If your laptop is slow, set `NUM_EPOCHS = 10`.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. criterion = nn.MSELoss(), optimizer = optim.Adam(..., lr=1e-3)
# 2. Train for at most NUM_EPOCHS=20 with early stopping, PATIENCE=5
# 3. For each epoch: train loop and validation loop (normal validation windows)
# 4. Track train_losses and val_losses; keep best_state = copy.deepcopy(model.state_dict())
#    and best_epoch when the validation loss improves
# 5. After the loop, restore the best weights: model.load_state_dict(best_state)

In [ ]:
# YOUR CODE HERE
# Hint: Plot train_losses and val_losses; mark the chosen epoch with plt.axvline(best_epoch)

---
## 8. Set the Threshold on the Validation Period

The anomaly score of a window is its mean squared reconstruction error. The threshold is the 99th percentile of the errors of **normal validation windows** -- data the model never trained on -- so about 1% of normal validation windows exceed it by construction. The percentile is fixed before the test period is scored. For comparison, the cell also prints the same percentile of the training-window errors: windows the model was fitted to are not a fair reference for new data. Finally, it shows where the normal validation windows above the threshold sit: mostly within two hours after a gap in the data, when the machine restarts after downtime and start-up transients reconstruct badly. Keep this in mind when you read the test results.

In [ ]:
# YOUR CODE HERE
# Hint:
# 1. reconstruction_errors(model, windows): model.eval(), torch.no_grad(), batches of 512;
#    per-window MSE: ((reconstructed - batch) ** 2).mean(dim=(1, 2)); return one array
# 2. THRESHOLD_PERCENTILE = 99
#    val_errors = reconstruction_errors(model, val_seq)
#    threshold = np.percentile(val_errors, THRESHOLD_PERCENTILE)
# 3. Print the median of val_errors, and for comparison the same percentile of the training-window errors
# 4. Optional: how long after the last gap in the data (usually a restart after downtime) do the
#    validation windows above the threshold sit?
#    restart_rows = np.flatnonzero(times.diff().to_numpy() > np.timedelta64(2, "m"))

---
## 9. Score the Test Period Once: Events, Misses and False Alarms

Operators do not see windows; they see **alerts**. An alert starts at the first window above the threshold after a quiet one and lasts while consecutive windows stay above it. We count:
- **events found**: breaks with an alert raised at most 30 minutes before the break, or at it; the lead time runs from the first such alert to the break;
- **events missed**: all other breaks (a few come too soon after a restart for a complete 60-minute window);
- **false alarms**: alerts that are neither such a warning nor raised in the hour after a break (while the window still contains it), also as a rate per week of operation.

The test period is scored once, with the threshold and all settings fixed above.

In [ ]:
test_mask = window_period == 2
test_ends = ends[test_mask]

# YOUR CODE HERE
# Hint:
# 1. test_errors = reconstruction_errors(model, sequences[test_mask]); flagged = test_errors > threshold
# 2. Alerts start at the first flagged window of each run (consecutive rows within one stretch):
#    continues = np.r_[False, flagged[:-1] & (np.diff(test_ends) == 1) & (np.diff(stretch[test_ends]) == 0)]
#    alert_rows = test_ends[flagged & ~continues]
# 3. For each test break e in test_events = event_rows[period[event_rows] == 2]:
#    - found if an alert lies in [e - LEAD_ROWS, e]; lead time = 2 * (e - first such alert) minutes
#    - an alert in (e, e + SEQ_LENGTH - 1] comes after the break (its window contains the break)
# 4. False alarms = alerts that are neither warnings nor after a break;
#    weeks = test_mask.sum() * 2 / (60 * 24 * 7) gives the rate per week
# 5. Print events found (and how many before the break itself, lead > 0), events missed,
#    false alarms (total and per week) and the median lead time

In [ ]:
# YOUR CODE HERE
# Hint (for contrast only): window-level precision, recall and F1, where a window is positive
# when its last row is a break: window_is_break = labels[test_ends] == 1
# Compare what this F1 tells an operator with the event counts above

---
## 10. Visualize Detected Anomalies

In [ ]:
# YOUR CODE HERE
# Hint: Create 2 subplots:
# Top: test-period reconstruction error over time (log scale) with the threshold line
#      and a vertical line at each break (times.to_numpy()[test_events])
# Bottom: histograms of the test errors of normal windows (normal[test_mask]) and of windows
#         in the 30 minutes before a break (warning_period[test_mask]), with the threshold

---
## 11. Summary

In this notebook we:

1. Loaded a multivariate process time series with rare sheet breaks (`y = 1`) from an Excel file.
2. Split it in time: training 1-14 May, validation 15-19 May, test 20-29 May.
3. Standardized the sensors with statistics of normal training rows only.
4. Built 60-minute windows inside contiguous stretches (no window crosses a gap or a split) and centred each window to remove slow drift.
5. Trained an LSTM autoencoder on normal training windows, with early stopping on normal validation windows.
6. Set the threshold at the 99th percentile of normal validation errors.
7. Scored the test period once: events found, events missed and false alarms per week, with a window-level F1 only for contrast.

**Key takeaways:**
- Autoencoders learn to reconstruct normal patterns. Anomalous data produces higher reconstruction error.
- Training on normal data only is critical -- including anomalies would teach the model to reconstruct them too.
- Fit the scaler on the training period and the threshold on a later normal period; errors on training windows are not a fair reference.
- The threshold trades missed events for false alarms; choose it from an alarm budget operators can live with.
- Caveat for this run: start-up transients after restarts set the 99th percentile of the normal validation errors, far above the bulk of normal errors, and most test breaks stay below it. A lower threshold would catch more breaks but also flag more of the test period, whose typical errors are higher than in validation (drift again). Excluding the first hours after a restart, or an alarm-budget threshold, would change the trade-off -- judge such changes on validation or a new period, not on this test set.
- Count events, lead times and false alarms per week: overlapping windows repeat one event many times, and a window-level F1 hides all three.
- Sensor levels drift between periods; without per-window centring a new regime looks anomalous. Retrain and re-threshold on a schedule.
- This approach is *unsupervised* in the sense that we do not need labelled anomalies for training, only for evaluation.